# MURL prompt-only baselines (Google Colab)

This notebook runs a reproducible prompt-only experiment against the frozen text-to-text benchmark. Repository scripts remain the source of truth for splitting, validation, and metrics; the notebook only orchestrates a Colab model run.

Before running: select a GPU runtime in **Runtime → Change runtime type**. The default model is small enough for most free Colab T4 sessions when loaded in 4-bit mode.

In [ ]:
# Experiment configuration — change only these values for a new run.
REPO_URL = 'https://github.com/cmkhoa/UIC-MURL-LLM.git'
BRANCH = 'main'
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
SPLIT = 'validation'                 # use validation while choosing a model/prompt
PROMPT_FILE = 'prompts/v0.1-zero-shot.md'
RUN_ID = 'qwen2.5-3b-zero-shot-v0.1-validation'
MAX_EXAMPLES = None                  # set, e.g., 10, for a smoke test
MAX_NEW_TOKENS = 256

assert SPLIT in {'train', 'validation', 'test'}
assert PROMPT_FILE in {'prompts/v0.1-zero-shot.md', 'prompts/v0.1-few-shot.md'}
assert RUN_ID and '/' not in RUN_ID and '\\' not in RUN_ID


In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path('/content/UIC-MURL-LLM')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())
subprocess.run(['python3', 'scripts/validate_benchmark.py', 'data/raw/graded-exams.jsonl', f'data/processed/{SPLIT}.jsonl'], check=True)


In [ ]:
%pip -q install 'transformers>=4.45' 'accelerate>=1.0' bitsandbytes sentencepiece

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, choose Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import json
from typing import Any

def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

def numbered(steps: list[str]) -> str:
    return '\n'.join(f'{index}. {step}' for index, step in enumerate(steps, 1))

def build_prompt(template: str, example: dict[str, Any]) -> str:
    replacements = {
        '{{problem}}': example['problem'],
        '{{reference_solution}}': numbered(example['reference_solution']),
        '{{student_solution}}': numbered(example['student_solution']),
    }
    for token, value in replacements.items():
        template = template.replace(token, value)
    if '{{' in template or '}}' in template:
        raise ValueError('Unfilled prompt placeholder')
    return template

examples = read_jsonl(f'data/processed/{SPLIT}.jsonl')
if MAX_EXAMPLES is not None:
    examples = examples[:MAX_EXAMPLES]
template = Path(PROMPT_FILE).read_text(encoding='utf-8')
print(f'Running {len(examples)} {SPLIT} examples with {MODEL_ID} and {PROMPT_FILE}.')
print(build_prompt(template, examples[0])[:800])


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quantization,
    torch_dtype=torch.bfloat16,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
def generate_output(prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': 'Follow the user instructions exactly. Return only the requested JSON object.'},
        {'role': 'user', 'content': prompt},
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

run_dir = Path('runs') / RUN_ID
run_dir.mkdir(parents=True, exist_ok=False)
outputs_path = run_dir / 'outputs.jsonl'
with outputs_path.open('w', encoding='utf-8') as handle:
    for index, example in enumerate(examples, 1):
        raw = generate_output(build_prompt(template, example))
        handle.write(json.dumps({'example_id': example['example_id'], 'output': raw}, ensure_ascii=False) + '\n')
        if index % 10 == 0 or index == len(examples):
            print(f'Completed {index}/{len(examples)}')
print('Saved outputs:', outputs_path)


In [ ]:
import shutil

results_dir = Path('results') / RUN_ID
if results_dir.exists():
    raise FileExistsError(f'Results already exist: {results_dir}. Use a new RUN_ID.')
subprocess.run([
    'python3', 'evaluation/evaluate.py',
    '--gold', f'data/processed/{SPLIT}.jsonl',
    '--outputs', str(outputs_path),
    '--run-id', RUN_ID,
], check=True)
aggregate = json.loads((results_dir / 'aggregate.json').read_text(encoding='utf-8'))
aggregate


## Record the run

Copy the aggregate metrics, model revision, Colab GPU, decoding settings, prompt version, dataset hash, and representative successes/failures into `docs/experiment-log-v0.1-baseline.md`. Do not use the test split until the model and prompt are frozen.

`runs/` and `results/` are intentionally ignored by Git. Download the outputs or save them to Google Drive before ending the Colab session.